# 06 — Demo
Run SignalRAG on a custom query, passages, and generated answer.

In [ ]:
import pickle
from signalrag import SignalExtractor
from signalrag.config import FEATURE_COLS


In [ ]:
# Load the best classifier (Random Forest)
with open("models/rf.pkl", "rb") as f:
    clf = pickle.load(f)

# Load SignalExtractor with the LLM so generation signals are real
# (requires a GPU; set load_llm=False to skip generation signals and use zeros)
try:
    extractor = SignalExtractor(load_llm=True)
    _has_llm  = True
    print("LLM loaded — all 17 signals will be computed")
except Exception as e:
    print(f"LLM not available ({e}). Generation signals will be set to 0.0.")
    extractor = SignalExtractor(load_llm=False)
    _has_llm  = False


In [ ]:
query    = "What is the capital of France?"
passages = ["France is a country in Western Europe. Its capital city is Paris."]
context  = " ".join(passages)

if _has_llm:
    # Full pipeline: LLM generates an answer and all 17 signals are extracted
    answer, signals = extractor.extract_all_signals(query, passages, context)
    print(f"Generated answer: {answer}")
else:
    # Fallback: supply your own answer; generation signals default to 0.0
    answer = "The capital of France is Paris."
    ret_sigs = extractor.extract_retrieval_signals(passages, query)
    ali_sigs = extractor.extract_alignment_signals(answer, context)
    gen_sigs = {c: 0.0 for c in FEATURE_COLS
                if c not in ret_sigs and c not in ali_sigs}
    signals  = {**ret_sigs, **gen_sigs, **ali_sigs}
    print("Note: generation signals are 0.0 (LLM not loaded). "
          "Reliability estimate may be less accurate.")

import numpy as np
X    = np.array([[signals[c] for c in clf.feat_cols]])
prob = clf.predict_proba(X)[0, 1]
pred = clf.predict(X)[0]
print(f"\nHallucination probability : {1 - prob:.3f}")
print(f"Predicted label           : {'FAITHFUL' if pred == 1 else 'HALLUCINATED'}")
